In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

In [2]:
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [6]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [9]:
print(df.horsepower.median())

254.0


In [16]:
np.random.seed(42)
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

target = 'fuel_efficiency_mpg'
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

# Split: 60% train, 20% validation, 20% test
train_df, temp_df = train_test_split(df_shuffled, test_size=0.4, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

y_train = train_df[target].values
y_val = val_df[target].values
y_test = test_df[target].values

In [27]:
hp_mean = train_df['horsepower'].mean()

# Option A: Fill with 0
X_train_zero = train_df[features].fillna(0)
X_val_zero = val_df[features].fillna(0)
model_zero = LinearRegression()
model_zero.fit(X_train_zero, y_train)
pred_zero = model_zero.predict(X_val_zero)
rmse_zero = np.sqrt(mean_squared_error(y_val, pred_zero))

# Option B: Fill with mean (from train set only)
X_train_mean = train_df[features].fillna(hp_mean)
X_val_mean = val_df[features].fillna(hp_mean)
model_mean = LinearRegression()
model_mean.fit(X_train_mean, y_train)
pred_mean = model_mean.predict(X_val_mean)
rmse_mean = np.sqrt(mean_squared_error(y_val, pred_mean))

print(f"RMSE (fill with 0):    {rmse_zero}")
print(f"RMSE (fill with mean): {rmse_mean}")

RMSE (fill with 0):    2.192211850344839
RMSE (fill with mean): 2.1887268604520287


In [28]:
X_train_reg = train_df[features].fillna(0)
X_val_reg = val_df[features].fillna(0)

alphas = [0, 0.01, 0.1, 1, 5, 10, 100]
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_reg, y_train)
    pred = model.predict(X_val_reg)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    print(f"alpha = {alpha} -> RMSE = {rmse}")

alpha = 0 -> RMSE = 2.1922118503448393
alpha = 0.01 -> RMSE = 2.1922118506078685
alpha = 0.1 -> RMSE = 2.19221185297514
alpha = 1 -> RMSE = 2.192211876648163
alpha = 5 -> RMSE = 2.1922119818683474
alpha = 10 -> RMSE = 2.192212113409086
alpha = 100 -> RMSE = 2.192214484087988


In [24]:
rmse_scores = []
for seed in range(10):
    train_s, temp_s = train_test_split(df_shuffled, test_size=0.4, random_state=seed)
    val_s, test_s = train_test_split(temp_s, test_size=0.5, random_state=seed)
    
    yt_s = train_s[target].values
    yv_s = val_s[target].values
    
    mean_s = train_s['horsepower'].mean()
    Xt_s = train_s[features].fillna(mean_s)
    Xv_s = val_s[features].fillna(mean_s)
    
    model = LinearRegression()
    model.fit(Xt_s, yt_s)
    pred_s = model.predict(Xv_s)
    rmse_s = np.sqrt(mean_squared_error(yv_s, pred_s))
    rmse_scores.append(rmse_s)

std_rmse = np.std(rmse_scores)
print(f"Standard Deviation of RMSE scores: {std_rmse}")

Standard Deviation of RMSE scores: 0.02061941075287982


In [29]:
train_val_df = pd.concat([train_df, val_df])
X_train_val = train_val_df[features].fillna(train_df['horsepower'].mean())
y_train_val = train_val_df[target].values

X_test = test_df[features].fillna(train_df['horsepower'].mean())
y_test_vals = test_df[target].values

final_model = LinearRegression()
final_model.fit(X_train_val, y_train_val)
test_preds = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test_vals, test_preds))

print(f"Final Test RMSE: {test_rmse}")

Final Test RMSE: 2.207217595716697
